In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
from urllib import request

In [4]:
request.urlretrieve("https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv", filename="ChatBotData.csv")

('ChatBotData.csv', <http.client.HTTPMessage at 0x7a352c0a08c0>)

In [5]:
import pandas as pd

df = pd.read_csv("/kaggle/working/ChatBotData.csv")

In [6]:
df

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0
...,...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!,2
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.,2
11820,흑기사 해주는 짝남.,설렜겠어요.,2
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.,2


In [7]:
import re 
def preprocess_sentence(sentence):
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = sentence.strip()
    return sentence

In [8]:
preprocess_sentence("안녕!!!")

'안녕 !  !  !'

In [9]:
questions = df['Q'].apply(preprocess_sentence).tolist()
answers = df['A'].apply(preprocess_sentence).tolist()

In [10]:
import tensorflow_datasets as tfds
tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    questions + answers, target_vocab_size=2**13)

2026-03-05 05:03:14.154176: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772686994.363070      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772686994.419818      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772686994.913835      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772686994.913890      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772686994.913893      55 computation_placer.cc:177] computation placer alr

In [11]:
START_TOKEN = [tokenizer.vocab_size]
END_TOKEN = [tokenizer.vocab_size + 1]
VOCAB_SIZE = tokenizer.vocab_size + 2
MAX_LENGTH = 40


In [12]:
tokenizer.encode(questions[2])

[7973, 1435, 4653, 7954, 3652, 67]

In [13]:
tmp = START_TOKEN + tokenizer.encode(questions[2]) + END_TOKEN

In [14]:
len(tmp)

8

In [15]:
if len(tmp) < MAX_LENGTH:
    tmp2 = (MAX_LENGTH - len(tmp)) * [0]

In [16]:
tmp + tmp2

[8178,
 7973,
 1435,
 4653,
 7954,
 3652,
 67,
 8179,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [17]:
def pad_seq(seq, maxlen=MAX_LENGTH):
    if len(seq) < maxlen:
        return seq + [0] * (maxlen - len(seq))
    return seq[:maxlen]
    
def tokenize_and_filter(inputs, outputs):
    tokenized_inputs, tokenized_outputs = [], []
    for (sentence1, sentence2) in zip(inputs, outputs):
        sentence1 = START_TOKEN + tokenizer.encode(sentence1) + END_TOKEN
        sentence2 = START_TOKEN + tokenizer.encode(sentence2) + END_TOKEN
        tokenized_inputs.append(sentence1)
        tokenized_outputs.append(sentence2)
    tokenized_inputs = [pad_seq(seq) for seq in tokenized_inputs]
    tokenized_outputs = [pad_seq(seq) for seq in tokenized_outputs]
    return np.array(tokenized_inputs), np.array(tokenized_outputs)


In [18]:
import numpy as np
questions, answers = tokenize_and_filter(questions, answers)

In [19]:
questions

array([[8178, 7915, 4207, ...,    0,    0,    0],
       [8178, 7971,   47, ...,    0,    0,    0],
       [8178, 7973, 1435, ...,    0,    0,    0],
       ...,
       [8178, 8159, 8079, ...,    0,    0,    0],
       [8178,  134,  166, ...,    0,    0,    0],
       [8178, 1953,  884, ...,    0,    0,    0]])

In [20]:
from torch.utils.data import Dataset, DataLoader

import torch
class ChatbotDataset(Dataset):
    def __init__(self, question, answer):
        self.question = question
        self.answer  = answer 

    def __len__(self):
        return len(self.question)

    def __getitem__(self, idx):
        return (torch.tensor(self.question[idx], dtype=torch.long), 
                torch.tensor(self.answer[idx][:-1], dtype=torch.long),
                torch.tensor(self.answer[idx][1:], dtype=torch.long))


In [21]:
data = ChatbotDataset(questions, answers)
a,b,c = next(iter(data))

In [22]:
c

tensor([3844,   74, 7894,    1, 8179,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0])

In [23]:
BATCH_SIZE = 64
data = ChatbotDataset(questions, answers)
dataloader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True)
test = next(iter(dataloader))

In [25]:
test[0].shape

torch.Size([64, 40])

In [29]:
import torch.nn as nn
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=9000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


In [30]:
def create_padding_mask(x):
    # (batch_size, 1, 1, seq_len)
    return (x == 0).unsqueeze(1).unsqueeze(2)


In [31]:
def create_look_ahead_mask(x):
    # (seq_len, seq_len)
    seq_len = x.size(1)
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    if x.is_cuda:
        mask = mask.cuda()
    return mask


In [32]:
def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, -1e9)
    p_attn = F.softmax(scores, dim=-1)
    return torch.matmul(p_attn, value), p_attn

In [33]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0
        self.depth = d_model // self.num_heads

        self.query_dense = nn.Linear(d_model, d_model)
        self.key_dense = nn.Linear(d_model, d_model)
        self.value_dense = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)
    
    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.transpose(1, 2)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)

        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        scaled_attention, _ = scaled_dot_product_attention(query, key, value, mask)
        scaled_attention = scaled_attention.transpose(1, 2).contiguous()
        concat_attention = scaled_attention.view(batch_size, -1, self.d_model)

        return self.dense(concat_attention)

In [34]:
class EncoderLayer(nn.Module):
    def __init__(self, dff, d_model, num_heads, dropout):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x ,  mask):
        attn_output = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2

In [35]:
class DecoderLayer(nn.Module):
    def __init__(self, dff, d_model, num_heads, dropout):
        super(DecoderLayer, self).__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm3 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, look_ahead_mask, padding_mask):
        attn1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1)
        out1 = self.layernorm1(x + attn1)

        attn2 = self.mha2(out1, enc_output, enc_output, padding_mask)
        attn2 = self.dropout2(attn2)
        out2 = self.layernorm2(out1 + attn2)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output)
        out3 = self.layernorm3(out2 + ffn_output)
        return out3

In [36]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, num_layers, dff, d_model, num_heads, dropout):
        super(Transformer, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        
        self.enc_layers = nn.ModuleList([EncoderLayer(dff, d_model, num_heads, dropout) for _ in range(num_layers)])
        self.dec_layers = nn.ModuleList([DecoderLayer(dff, d_model, num_heads, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, inputs, dec_inputs):
        # 1. 마스크 생성
        enc_padding_mask = create_padding_mask(inputs)
        dec_padding_mask = create_padding_mask(inputs)
        look_ahead_mask = torch.max(
            create_look_ahead_mask(dec_inputs),
            create_padding_mask(dec_inputs)
        )

        # 2. 인코더
        enc_out = self.embedding(inputs) * math.sqrt(self.d_model)
        enc_out = self.dropout(self.pos_encoding(enc_out))
        for layer in self.enc_layers:
            enc_out = layer(enc_out, enc_padding_mask)
        

        # 3. 디코더
        dec_out = self.embedding(dec_inputs) * math.sqrt(self.d_model)
        dec_out = self.dropout(self.pos_encoding(dec_out))
        for layer in self.dec_layers:
            dec_out = layer(dec_out, enc_out, look_ahead_mask, dec_padding_mask)

        return self.fc_out(dec_out)


In [37]:
NUM_LAYERS = 2
D_MODEL = 256
NUM_HEADS = 8
DFF = 512
DROPOUT = 0.1
EPOCHS = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [39]:
class CustomSchedule:
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = step + 1 # 0으로 나누는 것을 방지
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)


In [40]:
import math
model = Transformer(VOCAB_SIZE, NUM_LAYERS, DFF, D_MODEL, NUM_HEADS, DROPOUT).to(device)


In [41]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=CustomSchedule(D_MODEL))


In [42]:
import torch.nn.functional as F
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, dec_inputs, outputs) in enumerate(dataloader):
        inputs, dec_inputs, outputs = inputs.to(device), dec_inputs.to(device), outputs.to(device)
        
        optimizer.zero_grad()
        predictions = model(inputs, dec_inputs)
        
        # predictions shape: (batch_size, seq_len, vocab_size)
        # outputs shape: (batch_size, seq_len)
        loss = criterion(predictions.view(-1, VOCAB_SIZE), outputs.view(-1))
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(dataloader):.4f}")


Epoch 1/50 Loss: 7.9156
Epoch 2/50 Loss: 6.2245
Epoch 3/50 Loss: 5.6987
Epoch 4/50 Loss: 5.3986
Epoch 5/50 Loss: 5.0860
Epoch 6/50 Loss: 4.7639
Epoch 7/50 Loss: 4.4412
Epoch 8/50 Loss: 4.1113
Epoch 9/50 Loss: 3.7691
Epoch 10/50 Loss: 3.4111
Epoch 11/50 Loss: 3.0508
Epoch 12/50 Loss: 2.6848
Epoch 13/50 Loss: 2.3265
Epoch 14/50 Loss: 1.9801
Epoch 15/50 Loss: 1.6587
Epoch 16/50 Loss: 1.3596
Epoch 17/50 Loss: 1.1038
Epoch 18/50 Loss: 0.8825
Epoch 19/50 Loss: 0.7190
Epoch 20/50 Loss: 0.6017
Epoch 21/50 Loss: 0.5230
Epoch 22/50 Loss: 0.4694
Epoch 23/50 Loss: 0.4151
Epoch 24/50 Loss: 0.3628
Epoch 25/50 Loss: 0.3201
Epoch 26/50 Loss: 0.2843
Epoch 27/50 Loss: 0.2587
Epoch 28/50 Loss: 0.2420
Epoch 29/50 Loss: 0.2145
Epoch 30/50 Loss: 0.2029
Epoch 31/50 Loss: 0.1932
Epoch 32/50 Loss: 0.1751
Epoch 33/50 Loss: 0.1678
Epoch 34/50 Loss: 0.1602
Epoch 35/50 Loss: 0.1509
Epoch 36/50 Loss: 0.1407
Epoch 37/50 Loss: 0.1357
Epoch 38/50 Loss: 0.1272
Epoch 39/50 Loss: 0.1241
Epoch 40/50 Loss: 0.1132
Epoch 41/

In [43]:
def evaluate(sentence):
    sentence = preprocess_sentence(sentence)
    
    # 수정 포인트: START_TOKEN과 END_TOKEN을 감싸고 있던 대괄호 [] 제거
    sentence_tensor = torch.tensor(START_TOKEN + tokenizer.encode(sentence) + END_TOKEN).unsqueeze(0).to(device)
    output_tensor = torch.tensor(START_TOKEN).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        for i in range(MAX_LENGTH):
            predictions = model(sentence_tensor, output_tensor)
            
            # 현재(마지막) 시점의 예측 단어를 가져옵니다.
            predictions = predictions[:, -1:, :]
            predicted_id = torch.argmax(predictions, dim=-1)

            if predicted_id.item() == END_TOKEN[0]:
                break

            output_tensor = torch.cat([output_tensor, predicted_id], dim=-1)

    return output_tensor.squeeze(0).cpu().numpy()

def predict(sentence):
    prediction = evaluate(sentence)
    predicted_sentence = tokenizer.decode(
        [i for i in prediction if i < tokenizer.vocab_size])
    
    print('Input: {}'.format(sentence))
    print('Output: {}'.format(predicted_sentence))
    return predicted_sentence


In [55]:
predict("")

Input: 
Output: 많이 지쳤나봐요 .


'많이 지쳤나봐요 .'

In [56]:
torch.save(model.state_dict(), "./mymodel.pth")

In [57]:
dir(tokenizer)

['__abstractmethods__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_from_token_counts',
 '_byte_encode',
 '_cache_size',
 '_filename',
 '_id_to_subword',
 '_init_from_list',
 '_max_subword_len',
 '_read_lines_from_file',
 '_subword_to_id',
 '_subwords',
 '_token_to_ids',
 '_token_to_ids_cache',
 '_token_to_subwords',
 '_tokenizer',
 '_write_lines_to_file',
 'build_from_corpus',
 'decode',
 'encode',
 'load_from_file',
 'save_to_file',
 'subwords',
 'vocab_size']

In [58]:
tokenizer.save_to_file("./tokenizer")